In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import RobustScaler
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

Churn_client= pd.read_csv("Dataset_churn.csv")
Churn_client

# 1. Remplacer les 'unknown' par NaN
# 1. Remplacer les 'unknown' par NaN
df_str = Churn_client.astype(str).apply(lambda x: x.str.lower())
masque_unknown = df_str.apply(lambda x: x.str.contains('unknow', regex=False))

for col in Churn_client.columns:
    if col in masque_unknown.columns:
        Churn_client.loc[masque_unknown[col], col] = np.nan

# 2. Convertir et nettoyer les annotations scientifiques -> NaN
pattern_scientifique = r'^[+-]?\d+(\.\d+)?[eE][+-]?\d+$'
masque_sci = Churn_client.astype(str).apply(lambda x: x.str.match(pattern_scientifique))
cols_sci = masque_sci.sum()[masque_sci.sum() > 0].index.tolist()

for col in cols_sci:
    Churn_client[col] = pd.to_numeric(Churn_client[col], errors='coerce')

# 3. Imputation par le MODE pour les colonnes contenant initialement des 'unknown' (variables catégorielles)
cols_unknown = masque_unknown.sum()[masque_unknown.sum() > 0].index.tolist()
cols_categoriques = ['gadget_type', 'device_category', 'smart_phone_flag', 'manufacturer']

for col in cols_categoriques:
    # Calcul du mode en ignorant les NaN
    mode_val = Churn_client[col].dropna().mode()[0]
    # Remplacement des NaN par la valeur du mode
    Churn_client[col] = Churn_client[col].fillna(mode_val)
    
# 4. Imputation par la MÉDIANE pour les colonnes contenant des annotations scientifiques (variables numériques)
for col in cols_sci:
    if Churn_client[col].isnull().sum() > 0:
        mediane_val = Churn_client[col].median()
        Churn_client[col] = Churn_client[col].fillna(mediane_val)

print("Imputation terminée avec succès !")
print(f"NaN restants dans les colonnes à 'unknown' : {Churn_client[cols_unknown].isnull().sum().sum()}")

print(f"NaN restants dans les colonnes à annotations scientifiques : {Churn_client[cols_sci].isnull().sum().sum()}")



In [ ]:
 # Assurez-vous d'avoir scikit-learn à jour (>= 1.3)

# 1. Création de la cible basée sur mth2 (novembre - mois le plus récent)
# Churn (1) = 0 jour d'activité en mth2, Actif (0) = au moins 1 jour
Churn_client['churn'] = np.where(Churn_client['active_days_count_mth2'] == 0, 1, 0)

# 2. Séparation des features (X) et de la cible (y)
# ATTENTION : On ajoute 'active_days_count_mth2' dans la liste d'exclusion pour éviter le Data Leakage !
cols_to_drop = ['churn', 'msisdn', 'active_days_count_mth2']

# On garde uniquement les colonnes qui existent vraiment dans le dataset pour éviter les erreurs
cols_to_drop = [c for c in cols_to_drop if c in Churn_client.columns]

X = Churn_client.drop(columns=cols_to_drop)
y = Churn_client['churn'] # Correction ici : 'churn' au lieu de 'target'

# 3. Split Train / Test stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Target Encoding sur les variables catégorielles
cols_candidats = ['device_category', 'manufacturer', 'gadget_type', 'smart_phone_flag', 'value_segment']
cols_to_target_encode = [col for col in cols_candidats if col in X_train.columns]

# Encodage avec lissage automatique et validation croisée pour éviter l'overfitting
encoder = TargetEncoder(smooth="auto", cv=5)

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# Fit et Transform sur le Train
X_train_encoded[cols_to_target_encode] = encoder.fit_transform(X_train[cols_to_target_encode], y_train)
# Uniquement Transform sur le Test
X_test_encoded[cols_to_target_encode] = encoder.transform(X_test[cols_to_target_encode])

print(f"✓ Cible créée sur mth2 (Taux de churn : {y.mean():.2%})")
print(f"✓ Colonnes encodées : {cols_to_target_encode}")
print(f"✓ Forme de X_train_encoded : {X_train_encoded.shape}")

✓ Cible créée sur mth2 (Taux de churn : 6.46%)
✓ Colonnes encodées : ['device_category', 'manufacturer', 'gadget_type', 'smart_phone_flag', 'value_segment']
✓ Forme de X_train_encoded : (73091, 60)


In [ ]:
# 1. Copie des DataFrames encodés pour conserver une sauvegarde
X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

# 2. Identification des colonnes numériques à standardiser
# (On peut standardiser toutes les colonnes de X car elles sont maintenant toutes numériques)
cols_to_scale = X_train_encoded.columns.tolist()

# 3. Initialisation du RobustScaler
scaler = RobustScaler()

# 4. Fitting uniquement sur X_train et transformation de X_train et X_test
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train_encoded[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test_encoded[cols_to_scale])

print("✓ Standardisation (RobustScaler) appliquée avec succès !")
print(f"✓ Forme de X_train_scaled : {X_train_scaled.shape}")
print(f"✓ Forme de X_test_scaled  : {X_test_scaled.shape}")

✓ Standardisation (RobustScaler) appliquée avec succès !
✓ Forme de X_train_scaled : (73091, 60)
✓ Forme de X_test_scaled  : (18273, 60)


In [ ]:
from imblearn.over_sampling import SMOTE

# Application de SMOTE uniquement sur le jeu d'ENTRAÎNEMENT
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Distribution initiale dans y_train : \n{y_train.value_counts()}")
print(f"\nNouvelle distribution après SMOTE : \n{y_train_resampled.value_counts()}")
print("\nDimensions :")

print(
    "Avant :",
    X_train_scaled.shape
)

print(
    "Après :",
    X_train_resampled.shape
)

Distribution initiale dans y_train : 
churn
0    68373
1     4718
Name: count, dtype: int64

Nouvelle distribution après SMOTE : 
churn
0    68373
1    68373
Name: count, dtype: int64

Dimensions :
Avant : (73091, 60)
Après : (136746, 60)
